# Evaluacion Transversal — Predicción de abandono de clientes en ventas y marketing


**Integrantes:**

- Sebastian Lagos
- Oscar Oreste

**Contexto**

Una empresa del área de ventas y marketing dispone de información demográfica,
comercial y de experiencia de sus clientes. El objetivo del proyecto es analizar
estos datos y desarrollar un modelo de clasificación que permita identificar
clientes con riesgo de abandono.

La unidad de análisis corresponde a cada cliente y la variable objetivo es
`churn`, donde:

- `0`: el cliente no abandonó.
- `1`: el cliente abandonó.

El resultado puede apoyar decisiones de retención, marketing y experiencia del
cliente.

## Objetivos del proyecto

### Objetivo general

Desarrollar una solución reproducible de ciencia de datos para identificar
clientes con riesgo de abandono mediante técnicas de limpieza, integración de
datos, análisis exploratorio, clasificación y visualización interactiva.

### Objetivos específicos

1. Auditar la estructura y calidad del dataset.
2. Aplicar reglas de limpieza reproducibles sin modificar el archivo original.
3. Integrar indicadores externos mediante una API.
4. Analizar las características asociadas al abandono.
5. Comparar modelos de clasificación mediante accuracy.
6. Interpretar los resultados utilizando una matriz de confusión.
7. Construir un dashboard interactivo con Plotly 

## 1. Importación de librerías

Se importan las librerías principales que se utilizarán para cargar, revisar y
trabajar con el dataset.

Las librerías de visualización, API y Machine Learning se incorporarán en las
secciones correspondientes.

In [37]:
from pathlib import Path

import numpy as np
import pandas as pd

from IPython.display import display

RANDOM_STATE = 42

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:,.2f}".format)

## 2. Carga del dataset

El dataset se encuentra almacenado en la carpeta `data/raw`.

Se utilizan dos rutas posibles para permitir que el notebook pueda ejecutarse
desde la raíz del repositorio o desde la carpeta `notebook`.

In [38]:
rutas_posibles = [
    Path("data/raw/sales_marketing_customer_dataset.csv"),
    Path("../data/raw/sales_marketing_customer_dataset.csv")
]

ruta_dataset = next(
    (ruta for ruta in rutas_posibles if ruta.exists()),
    None
)

if ruta_dataset is None:
    raise FileNotFoundError(
        "No se encontró sales_marketing_customer_dataset.csv en data/raw."
    )

df = pd.read_csv(ruta_dataset)

print("Dataset cargado correctamente.")
print("Ruta utilizada:", ruta_dataset)

Dataset cargado correctamente.
Ruta utilizada: ..\data\raw\sales_marketing_customer_dataset.csv


## 3. Vista inicial del dataset

Se muestran las primeras filas para conocer la estructura general y observar
ejemplos de los datos disponibles.

In [39]:
df.head()

,customer_id,gender,age,country,city,signup_date,last_purchase_date,acquisition_channel,device_type,subscription_type,is_premium_user,total_visits,avg_session_time,pages_per_session,email_open_rate,email_click_rate,total_spent,avg_order_value,discount_used,coupon_code,support_tickets,refund_requested,delivery_delay_days,payment_method,satisfaction_score,nps_score,marketing_spend_per_user,lifetime_value,last_3_month_purchase_freq,churn
0,10001,Male,52.00,India,Berlin,2022-05-10 00:00:00,2024-12-31 00:00:00,Email,Tablet,Annual,1,7,13.90,5.42,0.67,0.26,559.52,65.25,0,NEW20,0,0,3,UPI,3.00,10,27.56,915.31,14,0
1,10002,NaN,35.00,Germany,Mumbai,2024-06-16 00:00:00,2024-05-07 00:00:00,Organic,Desktop,Monthly,0,19,5.11,5.35,0.70,0.37,356.49,48.47,1,NEW20,5,0,3,BKash,3.00,7,15.15,"2,079.96",11,0
2,10003,Female,27.00,Germany,London,2023-08-23 00:00:00,2024-04-28 00:00:00,Email,Mobile,Annual,1,18,9.74,3.59,0.47,0.44,689.33,77.82,0,NaN,1,0,2,UPI,5.00,6,13.51,"1,379.15",9,0
3,10004,Female,36.00,India,Mumbai,2024-01-28 00:00:00,2023-05-20 00:00:00,Facebook Ads,Tablet,Annual,1,16,9.64,2.95,0.58,0.37,445.43,71.71,0,NaN,0,0,2,PayPal,4.00,6,25.65,774.65,7,0
4,10005,Male,29.00,USA,Hamburg,2023-07-21 00:00:00,2024-04-07 00:00:00,Referral,Mobile,Monthly,0,12,7.79,2.41,0.05,0.16,686.29,44.99,1,NaN,2,1,4,BKash,3.00,1,12.39,87.68,11,0


## 4. Inspección inicial del dataset

Se revisan las dimensiones, los nombres de las columnas y los tipos de datos.

Esta inspección permite comprender la estructura antes de aplicar cualquier
limpieza o transformación.

In [40]:
print("Dimensión del dataset (filas, columnas):")
print(df.shape)

print("\nColumnas del dataset:")
print(df.columns.tolist())

print("\nInformación general:")
df.info()

Dimensión del dataset (filas, columnas):
(15000, 30)

Columnas del dataset:
['customer_id', 'gender', 'age', 'country', 'city', 'signup_date', 'last_purchase_date', 'acquisition_channel', 'device_type', 'subscription_type', 'is_premium_user', 'total_visits', 'avg_session_time', 'pages_per_session', 'email_open_rate', 'email_click_rate', 'total_spent', 'avg_order_value', 'discount_used', 'coupon_code', 'support_tickets', 'refund_requested', 'delivery_delay_days', 'payment_method', 'satisfaction_score', 'nps_score', 'marketing_spend_per_user', 'lifetime_value', 'last_3_month_purchase_freq', 'churn']

Información general:
<class 'pandas.DataFrame'>
RangeIndex: 15000 entries, 0 to 14999
Data columns (total 30 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   customer_id                 15000 non-null  int64  
 1   gender                      14262 non-null  str    
 2   age                         13800 non-null

## 5. Resumen estadístico inicial

Se calculan estadísticas descriptivas para las variables numéricas.

Esto permite observar sus valores mínimos, máximos, promedios y posibles
diferencias de escala.

In [41]:
print("Resumen estadístico de las columnas numéricas:")
display(df.describe().T)

Resumen estadístico de las columnas numéricas:


,count,mean,std,min,25%,50%,75%,max
customer_id,"15,000.00","17,500.50","4,330.27","10,001.00","13,750.75","17,500.50","21,250.25","25,000.00"
age,"13,800.00",35.20,10.33,-4.00,28.00,35.00,42.00,95.00
is_premium_user,"15,000.00",0.30,0.46,0.00,0.00,0.00,1.00,1.00
total_visits,"15,000.00",15.00,3.89,3.00,12.00,15.00,18.00,31.00
avg_session_time,"15,000.00",8.02,2.99,0.01,5.97,7.99,10.06,19.12
pages_per_session,"15,000.00",4.00,1.48,0.01,2.99,4.00,5.01,10.84
email_open_rate,"15,000.00",0.50,0.29,0.00,0.24,0.50,0.75,1.00
email_click_rate,"15,000.00",0.25,0.14,0.00,0.13,0.25,0.38,0.50
total_spent,"13,950.00",524.36,467.05,0.27,300.43,498.84,702.40,"15,910.43"
avg_order_value,"15,000.00",60.08,24.75,0.07,43.03,60.11,76.89,154.55


## 6. Revisión del identificador de clientes

La columna `customer_id` identifica a cada cliente. Se comprueba su cantidad de
valores únicos y si existen identificadores repetidos.

In [42]:
print("Cantidad de registros:", len(df))
print("Clientes únicos:", df["customer_id"].nunique())
print(
    "Identificadores duplicados:",
    df["customer_id"].duplicated().sum()
)

Cantidad de registros: 15000
Clientes únicos: 15000
Identificadores duplicados: 0


## 7. Distribución de la variable objetivo

La variable `churn` indica si un cliente abandonó o no la empresa:

- `0`: no abandonó;
- `1`: abandonó.

Se revisa la cantidad y el porcentaje de clientes pertenecientes a cada clase.

In [43]:
distribucion_churn = df["churn"].value_counts().sort_index()

porcentaje_churn = (
    df["churn"]
    .value_counts(normalize=True)
    .sort_index()
    .mul(100)
    .round(2)
)

resumen_churn = pd.DataFrame({
    "Cantidad": distribucion_churn,
    "Porcentaje": porcentaje_churn
})

display(resumen_churn)

,Cantidad,Porcentaje
churn,,
0,12702,84.68
1,2298,15.32


## 8. Baseline de accuracy

Como existe una clase mayoritaria, se calcula el resultado que obtendría una
regla que predijera siempre la clase más frecuente.

Este valor se utilizará posteriormente como referencia para evaluar los modelos
de clasificación.

In [44]:
clase_mayoritaria = df["churn"].mode()[0]

baseline_accuracy = (
    df["churn"] == clase_mayoritaria
).mean()

print("Clase mayoritaria:", clase_mayoritaria)
print(f"Baseline de accuracy: {baseline_accuracy:.2%}")

Clase mayoritaria: 0
Baseline de accuracy: 84.68%


### Interpretación inicial

El dataset contiene 15.000 registros y cada fila representa un cliente.

La clase `0`, correspondiente a clientes que no abandonaron la empresa, es
considerablemente más frecuente que la clase `1`.

Por este motivo, una accuracy cercana al 84,68 % no necesariamente representa
un buen modelo, ya que ese resultado puede alcanzarse prediciendo siempre la
clase mayoritaria.

Los modelos deberán superar o aportar información adicional respecto de este
baseline y sus resultados serán complementados con una matriz de confusión.

## 9. Revisión de calidad de los datos

Antes de limpiar el dataset se revisan los principales problemas de calidad:

- registros duplicados;
- valores nulos;
- categorías inconsistentes;
- edades inválidas;
- fechas incoherentes;
- relación entre país y ciudad;
- valores extremos.

En esta etapa solo se detectan y cuantifican los problemas. No se modifica el
DataFrame original.

## 10. Detección de duplicados

Se revisan primero los registros completamente duplicados.

También se comprueba si existen clientes con exactamente los mismos datos,
ignorando el identificador `customer_id`.

In [45]:
duplicados_exactos = df.duplicated().sum()

duplicados_sin_id = (
    df.drop(columns="customer_id")
    .duplicated()
    .sum()
)

print("Duplicados exactos:", duplicados_exactos)
print("Duplicados ignorando customer_id:", duplicados_sin_id)

Duplicados exactos: 0
Duplicados ignorando customer_id: 0


## 11. Detección de valores nulos

Se calcula la cantidad y el porcentaje de valores faltantes por columna.

El porcentaje permite evaluar si conviene imputar, conservar una categoría
explícita o excluir una variable del análisis.

In [46]:
resumen_nulos = pd.DataFrame({
    "Cantidad": df.isnull().sum(),
    "Porcentaje": (df.isnull().mean() * 100).round(2)
})

resumen_nulos = (
    resumen_nulos[resumen_nulos["Cantidad"] > 0]
    .sort_values("Cantidad", ascending=False)
)

display(resumen_nulos)

,Cantidad,Porcentaje
coupon_code,6133,40.89
age,1200,8.00
total_spent,1050,7.00
gender,738,4.92
satisfaction_score,702,4.68


### Interpretación de los valores nulos

Los valores faltantes no deben tratarse todos de la misma manera.

- En `coupon_code`, la ausencia no permite concluir si el cliente no utilizó un
  cupón o si el código simplemente no fue registrado. Por ello se conservará
  una categoría explícita de falta de registro y se creará una bandera de
  presencia del código.
- `gender` se conservará con una categoría explícita de información no
  registrada, sin inferir el género.
- `age`, `total_spent` y `satisfaction_score` son variables numéricas y su
  imputación se realizará posteriormente dentro del pipeline de Machine
  Learning.

No se utilizará el promedio o la mediana sobre el dataset completo, porque eso
podría introducir información del conjunto de prueba en el preprocesamiento.

## 12. Revisión de variables categóricas

Se inspeccionan las variables de texto con pocas categorías para detectar
diferencias de escritura, espacios adicionales o categorías vacías.

Las columnas de fechas se excluyen de esta revisión porque serán analizadas por
separado.

In [47]:
columnas_fecha = ["signup_date", "last_purchase_date"]

columnas_categoricas = [
    columna
    for columna in df.columns
    if columna not in columnas_fecha
    and pd.api.types.is_string_dtype(df[columna])
    and df[columna].nunique(dropna=True) <= 15
]

for columna in columnas_categoricas:
    print(f"\nVariable: {columna}")
    display(
        df[columna]
        .value_counts(dropna=False)
        .to_frame("Cantidad")
    )


Variable: gender


,Cantidad
gender,
Male,6844
Female,6686
NaN,738
Other,732



Variable: country


,Cantidad
country,
Germany,3072
India,3014
Bangladesh,2984
USA,2975
UK,2955



Variable: city


...

,Cantidad
subscription_type,
Monthly,7666
Annual,7334



Variable: coupon_code


,Cantidad
coupon_code,
NaN,6133
REF10,2995
SALE15,2986
NEW20,2886



Variable: payment_method


,Cantidad
payment_method,
UPI,3105
PayPal,3005
SEPA,2986
BKash,2971
Card,2933


## 13. Revisión de la edad

Se revisan los valores mínimos y máximos de `age`.

Las edades menores o iguales a cero se consideran claramente inválidas. Los
clientes menores de 18 o mayores de 80 se informan como observaciones, pero no
se eliminan automáticamente porque podrían ser registros reales.

In [48]:
resumen_edad = pd.DataFrame({
    "Indicador": [
        "Edad mínima",
        "Edad máxima",
        "Edades negativas",
        "Edad igual a cero",
        "Menores de 18 años",
        "Mayores de 80 años"
    ],
    "Cantidad o valor": [
        df["age"].min(),
        df["age"].max(),
        (df["age"] < 0).sum(),
        (df["age"] == 0).sum(),
        (df["age"] < 18).sum(),
        (df["age"] > 80).sum()
    ]
})

display(resumen_edad)

,Indicador,Cantidad o valor
0,Edad mínima,-4.00
1,Edad máxima,95.00
2,Edades negativas,3.00
3,Edad igual a cero,1.00
4,Menores de 18 años,516.00
5,Mayores de 80 años,30.00


### Interpretación de la edad

Solo las edades menores o iguales a cero se consideran inválidas sin requerir
una regla de negocio adicional.

Durante la limpieza, estos valores se convertirán en nulos. La imputación de la
edad se realizará posteriormente dentro del pipeline y se ajustará únicamente
con los datos de entrenamiento.

## 14. Revisión de fechas

Las columnas de fechas se convierten temporalmente para comprobar:

- formatos inválidos;
- fecha mínima y máxima;
- clientes cuya última compra aparece antes de la fecha de registro.

En esta etapa no se reemplazan ni se sobrescriben las columnas originales.

In [49]:
signup_date_temp = pd.to_datetime(
    df["signup_date"],
    errors="coerce"
)

last_purchase_date_temp = pd.to_datetime(
    df["last_purchase_date"],
    errors="coerce"
)

signup_invalidas = (
    df["signup_date"].notna()
    & signup_date_temp.isna()
).sum()

last_purchase_invalidas = (
    df["last_purchase_date"].notna()
    & last_purchase_date_temp.isna()
).sum()

secuencia_fecha_invalida = (
    signup_date_temp.notna()
    & last_purchase_date_temp.notna()
    & (last_purchase_date_temp < signup_date_temp)
)

print("Fechas de registro inválidas:", signup_invalidas)
print("Fechas de última compra inválidas:", last_purchase_invalidas)

print("\nRango de signup_date:")
print(signup_date_temp.min(), "a", signup_date_temp.max())

print("\nRango de last_purchase_date:")
print(last_purchase_date_temp.min(), "a", last_purchase_date_temp.max())

print(
    "\nÚltima compra anterior al registro:",
    secuencia_fecha_invalida.sum()
)

print(
    "Porcentaje de inconsistencias:",
    f"{secuencia_fecha_invalida.mean() * 100:.2f}%"
)

Fechas de registro inválidas: 0
Fechas de última compra inválidas: 0

Rango de signup_date:
2022-01-01 00:00:00 a 2024-09-26 00:00:00

Rango de last_purchase_date:
2023-01-01 00:00:00 a 2025-03-10 00:00:00

Última compra anterior al registro: 3762
Porcentaje de inconsistencias: 25.08%


### Interpretación de las fechas

Las fechas presentan un formato válido, pero existe una cantidad importante de
registros donde la última compra aparece antes de la fecha de inscripción.

Estas fechas no se corregirán inventando valores ni utilizando una fecha
promedio o mediana.

En la limpieza se creará una bandera de inconsistencia y la antigüedad solo se
calculará cuando el orden temporal sea válido.

## 15. Revisión de consistencia entre país y ciudad

Se revisa si una misma ciudad aparece asociada a diferentes países dentro del
dataset.

Para realizar una comprobación interna, se utiliza como referencia el país más
frecuente asociado a cada ciudad.

Esta regla permite detectar asociaciones inusuales, pero no constituye una
validación geográfica externa definitiva.

In [50]:
ubicaciones = (
    df[["country", "city"]]
    .dropna()
    .copy()
)

pais_frecuente_por_ciudad = (
    ubicaciones
    .groupby("city")["country"]
    .agg(lambda valores: valores.mode().iloc[0])
)

ubicaciones["pais_referencia_interno"] = (
    ubicaciones["city"]
    .map(pais_frecuente_por_ciudad)
)

ubicaciones["inconsistencia_interna"] = (
    ubicaciones["country"]
    != ubicaciones["pais_referencia_interno"]
)

cantidad_inconsistencias = (
    ubicaciones["inconsistencia_interna"]
    .sum()
)

porcentaje_inconsistencias = (
    ubicaciones["inconsistencia_interna"]
    .mean()
    * 100
)

print(
    "Combinaciones inusuales según la relación interna:",
    cantidad_inconsistencias
)

print(
    "Porcentaje:",
    f"{porcentaje_inconsistencias:.2f}%"
)


display(
    ubicaciones[
        ubicaciones["inconsistencia_interna"]
    ].head(10)
)

Combinaciones inusuales según la relación interna: 11818
Porcentaje: 78.79%


,country,city,pais_referencia_interno,inconsistencia_interna
2,Germany,London,USA,True
3,India,Mumbai,Germany,True
4,USA,Hamburg,UK,True
6,UK,New York,India,True
7,Germany,Berlin,India,True
9,Germany,Hamburg,UK,True
10,Germany,Delhi,USA,True
11,Bangladesh,New York,India,True
12,Germany,Hamburg,UK,True
13,India,Mumbai,Germany,True


### Interpretación de ubicación

La asociación dominante entre ciudad y país muestra una cantidad elevada de
combinaciones internas inusuales.

No se corregirá la ciudad mediante suposiciones. En el modelo inicial se
priorizará `country` y se evaluará excluir `city` debido a su inconsistencia.

La validación externa mediante la API se realizará por país, no por ciudad.

## 16. Revisión de valores extremos en total_spent

Se utiliza el rango intercuartílico para identificar valores extremos en
`total_spent`.

El método IQR define como posibles outliers los valores menores a:

Q1 - 1.5 × IQR

o mayores a:

Q3 + 1.5 × IQR

En esta etapa los registros solo se identifican. No se eliminan ni se truncan.

In [51]:
total_spent_sin_nulos = df["total_spent"].dropna()

q1_total_spent = total_spent_sin_nulos.quantile(0.25)
q3_total_spent = total_spent_sin_nulos.quantile(0.75)

iqr_total_spent = q3_total_spent - q1_total_spent

limite_inferior_total_spent = (
    q1_total_spent - 1.5 * iqr_total_spent
)

limite_superior_total_spent = (
    q3_total_spent + 1.5 * iqr_total_spent
)

filtro_outliers_total_spent = (
    df["total_spent"].notna()
    & (
        (df["total_spent"] < limite_inferior_total_spent)
        | (df["total_spent"] > limite_superior_total_spent)
    )
)

cantidad_outliers = filtro_outliers_total_spent.sum()

print("Mediana:", round(total_spent_sin_nulos.median(), 2))
print("Percentil 99:", round(total_spent_sin_nulos.quantile(0.99), 2))
print("Valor máximo:", round(total_spent_sin_nulos.max(), 2))

print("\nLímite inferior IQR:", round(limite_inferior_total_spent, 2))
print("Límite superior IQR:", round(limite_superior_total_spent, 2))

print("\nCantidad de valores extremos:", cantidad_outliers)
print(
    "Porcentaje:",
    f"{cantidad_outliers / len(df) * 100:.2f}%"
)

Mediana: 498.84
Percentil 99: 1216.01
Valor máximo: 15910.43

Límite inferior IQR: -302.51
Límite superior IQR: 1305.34

Cantidad de valores extremos: 78
Porcentaje: 0.52%


### Interpretación de los valores extremos

Los valores extremos de gasto no se eliminarán automáticamente, porque podrían
corresponder a clientes reales de alto valor.

En fases posteriores se evaluarán tres alternativas:

1. conservar la variable original;
2. crear una transformación logarítmica;
3. aplicar truncamiento únicamente si resulta necesario.

Cualquier límite utilizado para el modelo deberá calcularse con los datos de
entrenamiento y no con el dataset completo.

## 17. Revisión de variables constantes

Una variable constante contiene el mismo valor en todos los registros y no
aporta información para diferenciar clientes.

Se revisa si existen columnas con uno o ningún valor único.

In [52]:
cardinalidad = df.nunique(dropna=False)

variables_constantes = cardinalidad[
    cardinalidad <= 1
]

if variables_constantes.empty:
    print("No se detectaron variables constantes.")
else:
    print("Variables constantes detectadas:")
    display(variables_constantes.to_frame("Valores únicos"))

No se detectaron variables constantes.


## Conclusión de la auditoría de calidad

La auditoría permitió identificar problemas reales que justifican una etapa de
limpieza y transformación:

- existen valores nulos en variables numéricas y categóricas;
- se detectan edades menores o iguales a cero;
- las fechas tienen formato válido, pero existen secuencias temporales
  incoherentes;
- la relación entre ciudad y país presenta inconsistencias;
- `total_spent` contiene valores extremos;
- no se detectaron duplicados relevantes.

Durante esta etapa no se modificaron los datos.

En la siguiente fase se aplicarán reglas simples y justificadas:

- copia del DataFrame;
- conversión de edades inválidas en nulos;
- tratamiento explícito de categorías faltantes;
- conversión definitiva de fechas;
- creación de banderas de inconsistencia;
- creación de variables temporales válidas;
- preparación de las variables numéricas para su imputación posterior dentro
  del pipeline.

## 18. Limpieza e ingeniería inicial

A partir de la auditoría se crea una copia de trabajo llamada `df_clean`.

El DataFrame original `df` no se sobrescribe y el archivo ubicado en
`data/raw` permanece intacto. En esta fase se aplican únicamente reglas
determinísticas y justificadas; no se realiza imputación estadística, modelado,
integración de API ni análisis exploratorio completo.

In [53]:
df_clean = df.copy()

filas_antes = len(df_clean)
duplicados_antes = int(df_clean.duplicated().sum())

if duplicados_antes > 0:
    df_clean = df_clean.drop_duplicates().copy()

filas_despues_duplicados = len(df_clean)
duplicados_eliminados = filas_antes - filas_despues_duplicados

print("Filas antes de revisar duplicados:", filas_antes)
print("Duplicados exactos detectados:", duplicados_antes)
print("Duplicados exactos eliminados:", duplicados_eliminados)
print("Filas después de revisar duplicados:", filas_despues_duplicados)

Filas antes de revisar duplicados: 15000
Duplicados exactos detectados: 0
Duplicados exactos eliminados: 0
Filas después de revisar duplicados: 15000


### 18.1 Normalización de variables categóricas

Se eliminan espacios al inicio y al final de las variables de texto y las
cadenas vacías se convierten en valores faltantes.

La auditoría no mostró diferencias de escritura que justificaran cambiar
mayúsculas o minúsculas, por lo que no se recodifican categorías válidas.

En `gender` y `coupon_code` se utiliza la categoría `Sin registro` para
distinguir la ausencia de información. Esto no significa que se haya inferido
el género ni que un cliente necesariamente no haya utilizado un cupón.

In [54]:
columnas_texto_limpieza = [
    "gender",
    "country",
    "city",
    "acquisition_channel",
    "device_type",
    "subscription_type",
    "coupon_code",
    "payment_method"
]

for columna in columnas_texto_limpieza:
    df_clean[columna] = df_clean[columna].str.strip()
    df_clean[columna] = df_clean[columna].replace("", pd.NA)

gender_sin_registro_antes = int(df_clean["gender"].isna().sum())
coupon_sin_registro_antes = int(df_clean["coupon_code"].isna().sum())

df_clean["coupon_code_present_flag"] = (
    df_clean["coupon_code"].notna().astype(int)
)

df_clean["gender"] = df_clean["gender"].fillna("Sin registro")
df_clean["coupon_code"] = (
    df_clean["coupon_code"].fillna("Sin registro")
)

print("Gender sin registro identificados:", gender_sin_registro_antes)
print("Coupon code sin registro identificados:", coupon_sin_registro_antes)

print("\nCategorías finales de gender:")
display(df_clean["gender"].value_counts(dropna=False).to_frame("Cantidad"))

print("\nCategorías finales de coupon_code:")
display(df_clean["coupon_code"].value_counts(dropna=False).to_frame("Cantidad"))

Gender sin registro identificados: 738
Coupon code sin registro identificados: 6133

Categorías finales de gender:


,Cantidad
gender,
Male,6844
Female,6686
Sin registro,738
Other,732



Categorías finales de coupon_code:


,Cantidad
coupon_code,
Sin registro,6133
REF10,2995
SALE15,2986
NEW20,2886


### 18.2 Tratamiento de edades inválidas

Las edades menores o iguales a cero se consideran inválidas y se convierten en
`NaN`.

Los clientes menores de 18 años o mayores de 80 se conservan porque, sin una
regla de negocio adicional, no corresponde eliminarlos ni modificar sus
edades.

La imputación de `age` se realizará posteriormente dentro del pipeline y se
ajustará únicamente con los datos de entrenamiento.

In [55]:
edades_invalidas_antes = int((df_clean["age"] <= 0).sum())

df_clean.loc[df_clean["age"] <= 0, "age"] = np.nan

edades_invalidas_despues = int((df_clean["age"] <= 0).sum())

print("Edades menores o iguales a cero antes:", edades_invalidas_antes)
print("Edades menores o iguales a cero después:", edades_invalidas_despues)
print("Valores nulos actuales en age:", int(df_clean["age"].isna().sum()))

Edades menores o iguales a cero antes: 4
Edades menores o iguales a cero después: 0
Valores nulos actuales en age: 1204


### 18.3 Conversión de fechas y variables temporales

Las columnas `signup_date` y `last_purchase_date` se convierten de forma
definitiva a tipo fecha mediante `pd.to_datetime(..., errors="coerce")`.

Se crea `date_inconsistency_flag`:

- `1`: la última compra es anterior al registro;
- `0`: ambas fechas son válidas y el orden es correcto;
- valor faltante: no existen dos fechas válidas para evaluar la secuencia.

`tenure_days` se calcula únicamente cuando ambas fechas son válidas y la última
compra no es anterior al registro. Por ello, nunca se generan antigüedades
negativas.

In [56]:
df_clean["signup_date"] = pd.to_datetime(
    df_clean["signup_date"],
    errors="coerce"
)

df_clean["last_purchase_date"] = pd.to_datetime(
    df_clean["last_purchase_date"],
    errors="coerce"
)

ambas_fechas_validas = (
    df_clean["signup_date"].notna()
    & df_clean["last_purchase_date"].notna()
)

df_clean["date_inconsistency_flag"] = pd.Series(
    pd.NA,
    index=df_clean.index,
    dtype="Int64"
)

df_clean.loc[
    ambas_fechas_validas,
    "date_inconsistency_flag"
] = (
    df_clean.loc[
        ambas_fechas_validas,
        "last_purchase_date"
    ]
    < df_clean.loc[
        ambas_fechas_validas,
        "signup_date"
    ]
).astype(int)

secuencia_temporal_valida = (
    ambas_fechas_validas
    & (
        df_clean["last_purchase_date"]
        >= df_clean["signup_date"]
    )
)

df_clean["tenure_days"] = np.nan

df_clean.loc[
    secuencia_temporal_valida,
    "tenure_days"
] = (
    df_clean.loc[
        secuencia_temporal_valida,
        "last_purchase_date"
    ]
    - df_clean.loc[
        secuencia_temporal_valida,
        "signup_date"
    ]
).dt.days

df_clean["signup_year"] = (
    df_clean["signup_date"].dt.year.astype("Int64")
)

df_clean["last_purchase_year"] = (
    df_clean["last_purchase_date"].dt.year.astype("Int64")
)

print("Tipo de signup_date:", df_clean["signup_date"].dtype)
print("Tipo de last_purchase_date:", df_clean["last_purchase_date"].dtype)
print(
    "Inconsistencias temporales:",
    int(df_clean["date_inconsistency_flag"].eq(1).sum())
)
print(
    "Registros sin dos fechas válidas:",
    int(df_clean["date_inconsistency_flag"].isna().sum())
)
print(
    "Antigüedades negativas:",
    int((df_clean["tenure_days"] < 0).sum())
)

Tipo de signup_date: datetime64[us]
Tipo de last_purchase_date: datetime64[us]
Inconsistencias temporales: 3762
Registros sin dos fechas válidas: 0
Antigüedades negativas: 0


### 18.4 Bandera de inconsistencia de ubicación

No se corrigen ciudades ni se utiliza un diccionario geográfico inventado.

Se replica la regla interna de la auditoría: para cada ciudad se obtiene el país
más frecuente dentro del dataset y se marca con `1` cada asociación que no
coincide con esa referencia.

Esta bandera permite conservar el problema de calidad para auditoría, pero no
demuestra que una combinación sea geográficamente incorrecta. `country` será la
variable geográfica principal y `city` quedará fuera del primer modelo.

In [57]:
ubicacion_valida = (
    df_clean["country"].notna()
    & df_clean["city"].notna()
)

pais_referencia_por_ciudad = (
    df_clean.loc[ubicacion_valida]
    .groupby("city")["country"]
    .agg(lambda valores: valores.mode().iloc[0])
)

pais_referencia_filas = (
    df_clean.loc[ubicacion_valida, "city"]
    .map(pais_referencia_por_ciudad)
)

df_clean["location_inconsistency_flag"] = pd.Series(
    pd.NA,
    index=df_clean.index,
    dtype="Int64"
)

df_clean.loc[
    ubicacion_valida,
    "location_inconsistency_flag"
] = (
    df_clean.loc[ubicacion_valida, "country"]
    != pais_referencia_filas
).astype(int)

print(
    "Inconsistencias internas de ubicación:",
    int(df_clean["location_inconsistency_flag"].eq(1).sum())
)
print(
    "Registros sin ubicación evaluable:",
    int(df_clean["location_inconsistency_flag"].isna().sum())
)

Inconsistencias internas de ubicación: 11818
Registros sin ubicación evaluable: 0


### 18.5 Transformación de `total_spent`

Los valores extremos de gasto se conservan porque pueden representar clientes
reales de alto valor.

Como todos los valores observados de `total_spent` son no negativos, se crea
`total_spent_log` mediante `np.log1p`. Esta transformación reduce la asimetría
sin eliminar ni truncar clientes.

Los valores faltantes se mantienen como `NaN`. Cualquier imputación o
truncamiento futuro deberá ajustarse únicamente con el conjunto de
entrenamiento.

In [58]:
valores_negativos_total_spent = int(
    (df_clean["total_spent"].dropna() < 0).sum()
)

if valores_negativos_total_spent > 0:
    raise ValueError(
        "total_spent contiene valores negativos y no permite aplicar log1p "
        "sin una revisión adicional."
    )

df_clean["total_spent_log"] = np.log1p(
    df_clean["total_spent"]
)

print(
    "Valores negativos en total_spent:",
    valores_negativos_total_spent
)
print(
    "Nulos conservados en total_spent:",
    int(df_clean["total_spent"].isna().sum())
)
print(
    "Nulos conservados en total_spent_log:",
    int(df_clean["total_spent_log"].isna().sum())
)
print(
    "Máximo de total_spent:",
    round(df_clean["total_spent"].max(), 2)
)
print(
    "Máximo de total_spent_log:",
    round(df_clean["total_spent_log"].max(), 2)
)

Valores negativos en total_spent: 0
Nulos conservados en total_spent: 1050
Nulos conservados en total_spent_log: 1050
Máximo de total_spent: 15910.43
Máximo de total_spent_log: 9.67


### 18.6 Validación posterior a la limpieza

Se comprueba que:

- el número de filas solo cambie si existían duplicados exactos;
- el DataFrame original permanezca sin modificaciones;
- no existan edades menores o iguales a cero en `df_clean`;
- las fechas tengan el tipo correcto;
- `tenure_days` no contenga valores negativos;
- los nulos numéricos permanezcan disponibles para el pipeline;
- las nuevas variables hayan sido creadas.

In [59]:
columnas_nuevas = [
    "coupon_code_present_flag",
    "date_inconsistency_flag",
    "tenure_days",
    "signup_year",
    "last_purchase_year",
    "location_inconsistency_flag",
    "total_spent_log"
]

print("Dimensiones de df original:", df.shape)
print("Dimensiones de df_clean:", df_clean.shape)

print("\nDuplicados exactos antes:", duplicados_antes)
print(
    "Duplicados exactos después:",
    int(df_clean.duplicated().sum())
)

print(
    "\nEdades <= 0 conservadas en df original:",
    int((df["age"] <= 0).sum())
)
print(
    "Edades <= 0 en df_clean:",
    int((df_clean["age"] <= 0).sum())
)

print("\nTipos de fecha:")
print(df_clean[["signup_date", "last_purchase_date"]].dtypes)

print(
    "\nInconsistencias temporales:",
    int(df_clean["date_inconsistency_flag"].eq(1).sum())
)
print(
    "Inconsistencias internas de ubicación:",
    int(df_clean["location_inconsistency_flag"].eq(1).sum())
)
print(
    "Antigüedades negativas:",
    int((df_clean["tenure_days"] < 0).sum())
)

print("\nColumnas nuevas:")
print(columnas_nuevas)

print("\nValores nulos restantes:")
nulos_restantes = (
    df_clean.isna().sum()
    .loc[lambda serie: serie > 0]
    .sort_values(ascending=False)
    .to_frame("Cantidad")
)
display(nulos_restantes)

Dimensiones de df original: (15000, 30)
Dimensiones de df_clean: (15000, 37)

Duplicados exactos antes: 0


Duplicados exactos después: 0

Edades <= 0 conservadas en df original: 4
Edades <= 0 en df_clean: 0

Tipos de fecha:
signup_date           datetime64[us]
last_purchase_date    datetime64[us]
dtype: object

Inconsistencias temporales: 3762
Inconsistencias internas de ubicación: 11818
Antigüedades negativas: 0

Columnas nuevas:
['coupon_code_present_flag', 'date_inconsistency_flag', 'tenure_days', 'signup_year', 'last_purchase_year', 'location_inconsistency_flag', 'total_spent_log']

Valores nulos restantes:


,Cantidad
tenure_days,3762
age,1204
total_spent,1050
total_spent_log,1050
satisfaction_score,702


### 18.7 Exportación del dataset intermedio

El resultado de la limpieza se guarda en
`data/interim/customers_clean.csv`.

El archivo original se mantiene sin modificaciones. Este dataset intermedio
servirá como base para incorporar posteriormente información externa y
continuar con el análisis del abandono de clientes.

In [60]:
ruta_raiz_proyecto = ruta_dataset.parents[2]
carpeta_interim = ruta_raiz_proyecto / "data" / "interim"
carpeta_interim.mkdir(parents=True, exist_ok=True)

ruta_dataset_intermedio = (
    carpeta_interim / "customers_clean.csv"
)

df_clean.to_csv(
    ruta_dataset_intermedio,
    index=False
)

dimensiones_exportadas = pd.read_csv(
    ruta_dataset_intermedio
).shape

print("Dataset intermedio exportado correctamente.")
print("Ruta:", ruta_dataset_intermedio)
print("Dimensiones exportadas:", dimensiones_exportadas)

Dataset intermedio exportado correctamente.
Ruta: ..\data\interim\customers_clean.csv
Dimensiones exportadas: (15000, 37)


## Conclusión de la limpieza e ingeniería inicial

La limpieza conserva los 15.000 clientes porque la auditoría no detectó
duplicados exactos.

Las cuatro edades menores o iguales a cero se transforman en valores faltantes,
sin modificar edades que requieren una política de negocio adicional.

Las 3.762 secuencias temporales incoherentes quedan identificadas mediante una
bandera y no generan antigüedades negativas.

La regla interna de país y ciudad marca 11.818 asociaciones inusuales. Esta
cantidad corresponde a la salida real de la auditoría vigente y debe
interpretarse como una inconsistencia interna, no como una validación geográfica
externa.

Los valores extremos de `total_spent` se conservan y se añade una versión
logarítmica. Las variables numéricas faltantes no se imputan en esta fase para
evitar fuga de información.

El resultado queda disponible en `data/interim/customers_clean.csv` para la
posterior integración de indicadores externos y las siguientes etapas del
análisis.

## 19. Integración de indicadores externos

Para complementar el análisis se incorporan dos indicadores de contexto
provenientes de la API V2 del Banco Mundial:

- porcentaje de población usuaria de Internet (`IT.NET.USER.ZS`);
- PIB per cápita en dólares estadounidenses corrientes (`NY.GDP.PCAP.CD`).

La integración se realiza por país y por el año de registro del cliente. Estos
indicadores describen el contexto nacional y no representan características
individuales ni demuestran causas del abandono.

### 19.1 Preparación de países, años y rutas

Se revisan los países y años presentes en el dataset limpio.

Los nombres de países se homologan a códigos ISO3 utilizados por el Banco
Mundial. Si aparece un país sin correspondencia, el proceso se detiene para
evitar uniones incompletas o asignaciones inventadas.

También se preparan las rutas para almacenar el caché de la API y el dataset
procesado.

In [61]:
from datetime import datetime, timezone

import requests

mapa_codigos_banco_mundial = {
    "Germany": "DEU",
    "India": "IND",
    "Bangladesh": "BGD",
    "USA": "USA",
    "UK": "GBR"
}

paises_dataset = sorted(
    df_clean["country"].dropna().unique().tolist()
)

paises_sin_codigo = sorted(
    set(paises_dataset)
    - set(mapa_codigos_banco_mundial)
)

if paises_sin_codigo:
    raise ValueError(
        "Existen países sin código del Banco Mundial: "
        f"{paises_sin_codigo}"
    )

anios_dataset = sorted(
    df_clean["signup_year"]
    .dropna()
    .astype(int)
    .unique()
    .tolist()
)

if not anios_dataset:
    raise ValueError(
        "No existen años válidos en signup_year."
    )

ruta_raiz_proyecto = ruta_dataset.parents[2]

carpeta_external = (
    ruta_raiz_proyecto / "data" / "external"
)
carpeta_processed = (
    ruta_raiz_proyecto / "data" / "processed"
)

carpeta_external.mkdir(parents=True, exist_ok=True)
carpeta_processed.mkdir(parents=True, exist_ok=True)

ruta_cache_banco_mundial = (
    carpeta_external / "world_bank_indicators.csv"
)
ruta_dataset_procesado = (
    carpeta_processed / "customer_churn_processed.csv"
)

tabla_homologacion = pd.DataFrame({
    "country": paises_dataset,
    "country_code": [
        mapa_codigos_banco_mundial[pais]
        for pais in paises_dataset
    ]
})

display(tabla_homologacion)

print("Años de registro disponibles:", anios_dataset)
print("Ruta del caché:", ruta_cache_banco_mundial)
print("Ruta del dataset procesado:", ruta_dataset_procesado)

,country,country_code
0,Bangladesh,BGD
1,Germany,DEU
2,India,IND
3,UK,GBR
4,USA,USA


Años de registro disponibles: [2022, 2023, 2024]
Ruta del caché: ..\data\external\world_bank_indicators.csv
Ruta del dataset procesado: ..\data\processed\customer_churn_processed.csv


### 19.2 Consulta controlada de la API

La consulta utiliza únicamente los países y años presentes en el dataset, sin
realizar solicitudes por cada cliente.

Para reducir el impacto de fallas temporales del servicio se utiliza una sesión
HTTP con reintentos automáticos y espera progresiva.

La función incorpora:

- tiempo máximo separado para conexión y lectura;
- reintentos ante fallas temporales de red o del servidor;
- validación del código HTTP;
- validación básica de la estructura JSON;
- registro de la fecha de consulta;
- conservación de los valores faltantes publicados por la fuente.

Si la fuente no responde después de los reintentos, el flujo intenta utilizar
el caché local disponible.

In [62]:
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry


sesion_banco_mundial = requests.Session()

sesion_banco_mundial.headers.update({
    "User-Agent": (
        "Proyecto-Universitario-Churn/1.0 "
        "(integracion-indicadores-banco-mundial)"
    )
})


configuracion_reintentos = Retry(
    total=3,
    connect=3,
    read=3,
    status=3,
    backoff_factor=2,
    status_forcelist=(
        429,
        500,
        502,
        503,
        504
    ),
    allowed_methods=frozenset(["GET"]),
    respect_retry_after_header=True,
    raise_on_status=False
)


adaptador_http = HTTPAdapter(
    max_retries=configuracion_reintentos
)


sesion_banco_mundial.mount(
    "https://",
    adaptador_http
)

sesion_banco_mundial.mount(
    "http://",
    adaptador_http
)


def consultar_indicador_banco_mundial(
    codigos_pais,
    codigo_indicador,
    anio_inicio,
    anio_fin,
    timeout_conexion=10,
    timeout_lectura=60
):
    if not codigos_pais:
        raise ValueError(
            "No se proporcionaron países para consultar."
        )

    codigos_consulta = ";".join(codigos_pais)

    url = (
        "https://api.worldbank.org/v2/country/"
        f"{codigos_consulta}/indicator/{codigo_indicador}"
    )

    parametros = {
        "format": "json",
        "date": f"{anio_inicio}:{anio_fin}",
        "per_page": 100,
        "source": 2
    }

    respuesta = sesion_banco_mundial.get(
        url,
        params=parametros,
        timeout=(
            timeout_conexion,
            timeout_lectura
        )
    )

    respuesta.raise_for_status()

    try:
        contenido = respuesta.json()
    except requests.exceptions.JSONDecodeError as error_json:
        raise ValueError(
            "La API respondió, pero el contenido no es JSON válido."
        ) from error_json

    estructura_valida = (
        isinstance(contenido, list)
        and len(contenido) >= 2
        and isinstance(contenido[0], dict)
        and isinstance(contenido[1], list)
    )

    if not estructura_valida:
        raise ValueError(
            "La API no devolvió la estructura JSON esperada."
        )

    metadata = contenido[0]

    fecha_consulta_utc = datetime.now(
        timezone.utc
    ).isoformat(timespec="seconds")

    registros = []

    for observacion in contenido[1]:
        codigo_pais = observacion.get(
            "countryiso3code"
        )

        anio = observacion.get("date")

        if not codigo_pais or anio is None:
            raise ValueError(
                "La respuesta contiene una observación "
                "sin país o año."
            )

        try:
            anio = int(anio)
        except (TypeError, ValueError) as error_anio:
            raise ValueError(
                "La API devolvió un año inválido."
            ) from error_anio

        if codigo_pais not in codigos_pais:
            continue

        if not anio_inicio <= anio <= anio_fin:
            continue

        registros.append({
            "country_code": codigo_pais,
            "world_bank_country": (
                observacion.get("country", {})
                .get("value")
            ),
            "indicator_year": anio,
            "indicator_code": codigo_indicador,
            "indicator_name": (
                observacion.get("indicator", {})
                .get("value")
            ),
            "indicator_value": observacion.get(
                "value"
            ),
            "api_last_updated": metadata.get(
                "lastupdated"
            ),
            "query_utc": fecha_consulta_utc
        })

    if not registros:
        raise ValueError(
            "No se recibieron observaciones para el indicador "
            f"{codigo_indicador}."
        )

    resultado = pd.DataFrame(registros)

    resultado["indicator_value"] = pd.to_numeric(
        resultado["indicator_value"],
        errors="coerce"
    )

    if resultado.duplicated(
        [
            "country_code",
            "indicator_year"
        ]
    ).any():
        raise ValueError(
            "La API devolvió más de una observación "
            "para el mismo país y año."
        )

    resultado = (
        resultado
        .sort_values(
            [
                "country_code",
                "indicator_year"
            ]
        )
        .reset_index(drop=True)
    )

    return resultado

### 19.3 Descarga y caché local

Se consultan el porcentaje de población usuaria de Internet y el PIB per cápita
para las combinaciones de país y año requeridas.

Cuando las respuestas son válidas, se construye una tabla con una fila por país
y año y se guarda en `data/external/world_bank_indicators.csv`.

Antes de guardar el archivo se comprueba que:

- estén presentes las columnas esperadas;
- cada combinación de país y año sea única;
- el caché cubra todas las combinaciones solicitadas;
- los años y los indicadores tengan tipos válidos.

Si la consulta falla, se utiliza el caché local existente. Cuando no existe un
caché válido, el proceso se detiene en lugar de inventar o completar
indicadores.

In [63]:
codigos_consulta = sorted(
    mapa_codigos_banco_mundial.values()
)

anio_inicio = min(anios_dataset)
anio_fin = max(anios_dataset)


columnas_cache_esperadas = [
    "country",
    "country_code",
    "indicator_year",
    "internet_users_pct",
    "gdp_per_capita_usd",
    "internet_api_last_updated",
    "gdp_api_last_updated",
    "internet_query_utc",
    "gdp_query_utc"
]


codigo_a_pais = {
    codigo: pais
    for pais, codigo
    in mapa_codigos_banco_mundial.items()
}


grilla_pais_anio = (
    pd.MultiIndex.from_product(
        [
            codigos_consulta,
            anios_dataset
        ],
        names=[
            "country_code",
            "indicator_year"
        ]
    )
    .to_frame(index=False)
)


grilla_pais_anio["country"] = (
    grilla_pais_anio["country_code"]
    .map(codigo_a_pais)
)


try:
    datos_internet = consultar_indicador_banco_mundial(
        codigos_pais=codigos_consulta,
        codigo_indicador="IT.NET.USER.ZS",
        anio_inicio=anio_inicio,
        anio_fin=anio_fin
    )

    datos_pib = consultar_indicador_banco_mundial(
        codigos_pais=codigos_consulta,
        codigo_indicador="NY.GDP.PCAP.CD",
        anio_inicio=anio_inicio,
        anio_fin=anio_fin
    )

    internet_seleccionado = (
        datos_internet[
            [
                "country_code",
                "indicator_year",
                "indicator_value",
                "api_last_updated",
                "query_utc"
            ]
        ]
        .rename(columns={
            "indicator_value": (
                "internet_users_pct"
            ),
            "api_last_updated": (
                "internet_api_last_updated"
            ),
            "query_utc": (
                "internet_query_utc"
            )
        })
    )

    pib_seleccionado = (
        datos_pib[
            [
                "country_code",
                "indicator_year",
                "indicator_value",
                "api_last_updated",
                "query_utc"
            ]
        ]
        .rename(columns={
            "indicator_value": (
                "gdp_per_capita_usd"
            ),
            "api_last_updated": (
                "gdp_api_last_updated"
            ),
            "query_utc": (
                "gdp_query_utc"
            )
        })
    )

    indicadores_banco_mundial = (
        grilla_pais_anio
        .merge(
            internet_seleccionado,
            on=[
                "country_code",
                "indicator_year"
            ],
            how="left",
            validate="one_to_one"
        )
        .merge(
            pib_seleccionado,
            on=[
                "country_code",
                "indicator_year"
            ],
            how="left",
            validate="one_to_one"
        )
    )

    origen_indicadores = (
        "API del Banco Mundial"
    )

except (
    requests.RequestException,
    ValueError,
    KeyError,
    TypeError
) as error_api:
    print(
        "No fue posible obtener los indicadores "
        "desde la API."
    )

    print(
        "Detalle:",
        type(error_api).__name__,
        "-",
        str(error_api)
    )

    if not ruta_cache_banco_mundial.exists():
        raise RuntimeError(
            "La consulta al Banco Mundial falló y todavía "
            "no existe un caché local. Reintenta con conexión "
            "a Internet antes de continuar."
        ) from error_api

    try:
        indicadores_banco_mundial = pd.read_csv(
            ruta_cache_banco_mundial
        )
    except (
        OSError,
        pd.errors.EmptyDataError,
        pd.errors.ParserError
    ) as error_cache:
        raise RuntimeError(
            "Existe un archivo de caché, pero no pudo "
            "ser leído correctamente."
        ) from error_cache

    origen_indicadores = "caché local"


columnas_faltantes_cache = sorted(
    set(columnas_cache_esperadas)
    - set(indicadores_banco_mundial.columns)
)


if columnas_faltantes_cache:
    raise ValueError(
        "El archivo externo no contiene las columnas "
        f"esperadas: {columnas_faltantes_cache}"
    )


indicadores_banco_mundial[
    "indicator_year"
] = pd.to_numeric(
    indicadores_banco_mundial[
        "indicator_year"
    ],
    errors="raise"
).astype("Int64")


indicadores_banco_mundial[
    "internet_users_pct"
] = pd.to_numeric(
    indicadores_banco_mundial[
        "internet_users_pct"
    ],
    errors="coerce"
)


indicadores_banco_mundial[
    "gdp_per_capita_usd"
] = pd.to_numeric(
    indicadores_banco_mundial[
        "gdp_per_capita_usd"
    ],
    errors="coerce"
)


indicadores_banco_mundial = (
    indicadores_banco_mundial.loc[
        indicadores_banco_mundial[
            "country_code"
        ].isin(codigos_consulta)
        & indicadores_banco_mundial[
            "indicator_year"
        ].isin(anios_dataset)
    ]
    .copy()
)


if indicadores_banco_mundial.duplicated(
    [
        "country_code",
        "indicator_year"
    ]
).any():
    raise ValueError(
        "El archivo externo contiene países y años duplicados."
    )


combinaciones_esperadas = set(
    grilla_pais_anio[
        [
            "country_code",
            "indicator_year"
        ]
    ]
    .itertuples(
        index=False,
        name=None
    )
)


combinaciones_obtenidas = set(
    indicadores_banco_mundial[
        [
            "country_code",
            "indicator_year"
        ]
    ]
    .itertuples(
        index=False,
        name=None
    )
)


combinaciones_faltantes = sorted(
    combinaciones_esperadas
    - combinaciones_obtenidas
)


if combinaciones_faltantes:
    raise ValueError(
        "La información externa no cubre todas las "
        "combinaciones país-año requeridas: "
        f"{combinaciones_faltantes}"
    )


indicadores_banco_mundial = (
    indicadores_banco_mundial[
        columnas_cache_esperadas
    ]
    .sort_values(
        [
            "country",
            "indicator_year"
        ]
    )
    .reset_index(drop=True)
)


if origen_indicadores == "API del Banco Mundial":
    indicadores_banco_mundial.to_csv(
        ruta_cache_banco_mundial,
        index=False
    )


print("Fuente utilizada:", origen_indicadores)

print(
    "Dimensiones de los indicadores:",
    indicadores_banco_mundial.shape
)

print(
    "Ruta del caché:",
    ruta_cache_banco_mundial
)

display(indicadores_banco_mundial)

Fuente utilizada: API del Banco Mundial
Dimensiones de los indicadores: (15, 9)
Ruta del caché: ..\data\external\world_bank_indicators.csv


,country,country_code,indicator_year,internet_users_pct,gdp_per_capita_usd,internet_api_last_updated,gdp_api_last_updated,internet_query_utc,gdp_query_utc
0,Bangladesh,BGD,2022,41.62,"2,716.49",2026-07-13,2026-07-13,2026-07-15T10:11:59+00:00,2026-07-15T10:14:03+00:00
1,Bangladesh,BGD,2023,44.50,"2,551.02",2026-07-13,2026-07-13,2026-07-15T10:11:59+00:00,2026-07-15T10:14:03+00:00
2,Bangladesh,BGD,2024,53.42,"2,593.42",2026-07-13,2026-07-13,2026-07-15T10:11:59+00:00,2026-07-15T10:14:03+00:00
3,Germany,DEU,2022,91.63,"50,506.52",2026-07-13,2026-07-13,2026-07-15T10:11:59+00:00,2026-07-15T10:14:03+00:00
4,Germany,DEU,2023,92.48,"54,776.77",2026-07-13,2026-07-13,2026-07-15T10:11:59+00:00,2026-07-15T10:14:03+00:00
5,Germany,DEU,2024,93.50,"56,103.73",2026-07-13,2026-07-13,2026-07-15T10:11:59+00:00,2026-07-15T10:14:03+00:00
6,India,IND,2022,55.90,"2,279.98",2026-07-13,2026-07-13,2026-07-15T10:11:59+00:00,2026-07-15T10:14:03+00:00
7,India,IND,2023,60.25,"2,434.45",2026-07-13,2026-07-13,2026-07-15T10:11:59+00:00,2026-07-15T10:14:03+00:00
8,India,IND,2024,64.94,"2,591.99",2026-07-13,2026-07-13,2026-07-15T10:11:59+00:00,2026-07-15T10:14:03+00:00
9,UK,GBR,2022,95.47,"47,034.78",2026-07-13,2026-07-13,2026-07-15T10:11:59+00:00,2026-07-15T10:14:03+00:00


### 19.4 Revisión de disponibilidad

La tabla externa debe contener una combinación única por país y año.

Los valores faltantes se conservan porque algunos indicadores pueden publicarse
con retraso. No se reemplazan con el año anterior ni con promedios, ya que eso
alteraría la correspondencia temporal.

In [64]:
resumen_disponibilidad = (
    indicadores_banco_mundial
    .groupby("country", as_index=False)
    .agg(
        anios_consultados=(
            "indicator_year",
            "nunique"
        ),
        internet_disponible=(
            "internet_users_pct",
            lambda serie: int(serie.notna().sum())
        ),
        pib_disponible=(
            "gdp_per_capita_usd",
            lambda serie: int(serie.notna().sum())
        )
    )
)

print(
    "Combinaciones país-año esperadas:",
    len(codigos_consulta) * len(anios_dataset)
)

print(
    "Combinaciones país-año obtenidas:",
    len(indicadores_banco_mundial)
)

print(
    "Valores faltantes en uso de Internet:",
    int(
        indicadores_banco_mundial[
            "internet_users_pct"
        ].isna().sum()
    )
)

print(
    "Valores faltantes en PIB per cápita:",
    int(
        indicadores_banco_mundial[
            "gdp_per_capita_usd"
        ].isna().sum()
    )
)

display(resumen_disponibilidad)

Combinaciones país-año esperadas: 15
Combinaciones país-año obtenidas: 15
Valores faltantes en uso de Internet: 0
Valores faltantes en PIB per cápita: 0


,country,anios_consultados,internet_disponible,pib_disponible
0,Bangladesh,3,3,3
1,Germany,3,3,3
2,India,3,3,3
3,UK,3,3,3
4,USA,3,3,3


### 19.5 Unión con los clientes

Los indicadores se unen con `df_clean` mediante:

- `country_code`, derivado de `country`;
- `signup_year`, derivado de `signup_date`.

La unión es de tipo izquierda para conservar a todos los clientes. La validación
`many_to_one` garantiza que cada combinación de país y año corresponda como
máximo a una fila de indicadores.

In [65]:
df_processed = df_clean.copy()

df_processed["country_code"] = (
    df_processed["country"]
    .map(mapa_codigos_banco_mundial)
)

if df_processed["country_code"].isna().any():
    paises_sin_union = sorted(
        df_processed.loc[
            df_processed["country_code"].isna(),
            "country"
        ]
        .dropna()
        .unique()
        .tolist()
    )

    raise ValueError(
        "Existen países sin homologación: "
        f"{paises_sin_union}"
    )

filas_antes_union = len(df_processed)

df_processed = df_processed.merge(
    indicadores_banco_mundial[
        [
            "country_code",
            "indicator_year",
            "internet_users_pct",
            "gdp_per_capita_usd"
        ]
    ],
    left_on=["country_code", "signup_year"],
    right_on=["country_code", "indicator_year"],
    how="left",
    validate="many_to_one"
)

filas_despues_union = len(df_processed)

if filas_antes_union != filas_despues_union:
    raise ValueError(
        "La unión modificó la cantidad de clientes."
    )

combinaciones_sin_coincidencia = (
    df_processed.loc[
        df_processed["indicator_year"].isna(),
        ["country", "signup_year"]
    ]
    .drop_duplicates()
    .sort_values(["country", "signup_year"])
)

print("Filas antes de la unión:", filas_antes_union)
print("Filas después de la unión:", filas_despues_union)

print(
    "Clientes sin coincidencia país-año:",
    int(df_processed["indicator_year"].isna().sum())
)

print(
    "Clientes sin dato de uso de Internet:",
    int(df_processed["internet_users_pct"].isna().sum())
)

print(
    "Clientes sin dato de PIB per cápita:",
    int(df_processed["gdp_per_capita_usd"].isna().sum())
)

if not combinaciones_sin_coincidencia.empty:
    display(combinaciones_sin_coincidencia)

df_processed = df_processed.drop(
    columns="indicator_year"
)

Filas antes de la unión: 15000
Filas después de la unión: 15000
Clientes sin coincidencia país-año: 0
Clientes sin dato de uso de Internet: 0
Clientes sin dato de PIB per cápita: 0


### 19.6 Validación de la integración

Se verifica que:

- la cantidad de clientes no cambie;
- los identificadores permanezcan únicos;
- no existan países sin código;
- la unión no genere filas duplicadas;
- los indicadores externos conserven sus valores faltantes reales.

Los indicadores se mantendrán inicialmente como variables de contexto. Su
incorporación al modelo deberá justificarse y compararse posteriormente.

In [66]:
print("Dimensiones de df_clean:", df_clean.shape)
print("Dimensiones de df_processed:", df_processed.shape)

print(
    "Clientes únicos en df_processed:",
    df_processed["customer_id"].nunique()
)

print(
    "Identificadores duplicados:",
    int(
        df_processed[
            "customer_id"
        ].duplicated().sum()
    )
)

print(
    "Países sin código:",
    int(df_processed["country_code"].isna().sum())
)

columnas_externas = [
    "country_code",
    "internet_users_pct",
    "gdp_per_capita_usd"
]

print("\nColumnas externas incorporadas:")
print(columnas_externas)

print("\nResumen de indicadores externos:")
display(
    df_processed[
        [
            "internet_users_pct",
            "gdp_per_capita_usd"
        ]
    ].describe().T
)

Dimensiones de df_clean: (15000, 37)
Dimensiones de df_processed: (15000, 40)
Clientes únicos en df_processed: 15000
Identificadores duplicados: 0
Países sin código: 0

Columnas externas incorporadas:
['country_code', 'internet_users_pct', 'gdp_per_capita_usd']

Resumen de indicadores externos:


,count,mean,std,min,25%,50%,75%,max
internet_users_pct,"15,000.00",77.48,20.67,41.62,55.90,92.48,93.53,95.47
gdp_per_capita_usd,"15,000.00","38,006.06","31,078.98","2,279.98","2,591.99","49,919.69","56,103.73","86,169.66"


### 19.7 Exportación del dataset procesado

El dataset enriquecido se guarda en
`data/processed/customer_churn_processed.csv`.

El archivo conserva a todos los clientes y añade los indicadores externos sin
modificar los archivos ubicados en `data/raw` ni el dataset intermedio.

In [67]:
df_processed.to_csv(
    ruta_dataset_procesado,
    index=False
)

dimensiones_procesadas = pd.read_csv(
    ruta_dataset_procesado
).shape

print("Dataset procesado exportado correctamente.")
print("Ruta:", ruta_dataset_procesado)
print("Dimensiones exportadas:", dimensiones_procesadas)

print(
    "Caché externo disponible:",
    ruta_cache_banco_mundial.exists()
)

Dataset procesado exportado correctamente.
Ruta: ..\data\processed\customer_churn_processed.csv
Dimensiones exportadas: (15000, 40)
Caché externo disponible: True


## Conclusión de la integración de datos externos

Los clientes fueron enriquecidos mediante indicadores del Banco Mundial
consultados por país y año de registro.

La consulta se realiza por valores únicos y utiliza un caché local como respaldo
ante fallas de conexión. La unión conserva la cantidad original de clientes y
mantiene como faltantes los indicadores que la fuente no publica para una
combinación específica.

El porcentaje de población usuaria de Internet y el PIB per cápita representan
contexto nacional. Por ello, sus asociaciones con `churn` deberán interpretarse
con cautela y no como relaciones causales individuales.

El dataset procesado queda disponible en
`data/processed/customer_churn_processed.csv` para continuar con el análisis
exploratorio y la preparación del modelado.